In [0]:
%sql
SELECT * FROM detection.epg_station
WHERE inscape_station_name ILIKE '%newsmax%'
AND (ingested = 'TRUE' OR attributed = 'TRUE')
AND vendor_name = 'TIVO'

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from datetime import datetime, date

In [0]:
df_pre = spark.sql(
    f"""
    SELECT DATE(vc.session_start) AS session_day
    , SUM(vc.session_duration)/3600.0 AS ttl_duration
    FROM detection.viewing_content_firehose vc
    WHERE vc.session_start >= '2024-08-01 00:00:00'
      AND vc.session_start < '2025-04-18 00:00:00'
      AND vc.fk_zoo_id = 17
      AND vc.fk_station_id = 90856
      AND vc.partition_key >= '2024-08-01'
      AND vc.session_start < '2025-04-21'
    GROUP BY 1
 """)

In [0]:
df = df_pre.toPandas().fillna(0)

In [0]:
df.sort_values(by='session_day', inplace=True)

In [0]:
df.display()

Databricks visualization. Run in Databricks to view.

In [0]:
df.reset_index(drop=True, inplace=True)

In [0]:
df.head()

In [0]:
df['session_day'] = pd.to_datetime(df['session_day'])

In [0]:
ndf = df.copy()

In [0]:
df = ndf.copy()

In [0]:
df.loc[:, 'special_event'] = 0

df.loc[:, 'special_event'] = np.where(df.session_day == '2024-11-06', 3, df.special_event)
df.loc[:, 'special_event'] = np.where(df.session_day == '2025-01-20', 2, df.special_event)
df.loc[:, 'special_event'] = np.where(df.session_day == '2025-03-05', 1, df.special_event)

In [0]:
df.loc[df['session_day'] == '2024-12-17', 'ttl_duration'] = 0
df.loc[df['session_day'] == '2025-01-24', 'ttl_duration'] = 0
df.loc[df['session_day'] == '2025-01-25', 'ttl_duration'] = 0

In [0]:
df.loc[df['session_day'].dt.strftime('%Y-%m-%d').between('2025-02-25', '2025-03-03'), 'ttl_duration'] = 0

In [0]:
df.set_index('session_day', inplace=True)

df['dayofweek'] = df.index.dayofweek
df['day'] = df.index.day
df['month'] = df.index.month

df['dow_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)

In [0]:
df_known = df[df['ttl_duration'] > 0]
df_missing = df[df['ttl_duration'] == 0]

features = ['dayofweek', 'day', 'month', 'dow_sin', 'dow_cos', 'special_event']
X_train = df_known[features]
y_train = df_known['ttl_duration']

X_missing = df_missing[features]

In [0]:
df_missing

In [0]:
model = GradientBoostingRegressor(n_estimators=100, max_depth=7, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_missing)

In [0]:
df.loc[df_missing.index, 'ttl_duration_predicted'] = y_pred

In [0]:
df.head()

In [0]:
df[df.ttl_duration_predicted >= 0]

In [0]:
df['ttl_duration'] = pd.to_numeric(df['ttl_duration'], errors='coerce')
df['ttl_duration_predicted'] = pd.to_numeric(df['ttl_duration_predicted'], errors='coerce')

In [0]:
plt.figure(figsize=(15,5))
df['ttl_duration'].plot(label='Known', alpha=0.6)
df['ttl_duration_predicted'].plot(label='Predicted (Long Gap)', linestyle='--')
plt.legend()
plt.title('Total Duration for Fox News: Forecast for Missing Month')
plt.show()

In [0]:
pred_df = df[df['ttl_duration_predicted'].notnull()]

In [0]:
pred_df.drop(columns=['dayofweek', 'day', 'month', 'dow_sin', 'dow_cos'], inplace=True)

In [0]:
pred_df.reset_index(inplace=True)

In [0]:
pred_df

In [0]:
pred_df.columns

In [0]:
spark_df = spark.createDataFrame(pred_df)

In [0]:
spark_df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("dev.mohit_gangwani.newsmax_predicted_values_for_missing_period_innovid")

In [0]:
%sql
SELECT * FROM dev.mohit_gangwani.issue_with_innovid_foxnews_20250417_daily_wo_live
limit 100